<a href="https://colab.research.google.com/github/rodrigololr/ml/blob/main/Machine_learning_01_diogo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade: Predição de Diabetes

Nesta atividade, eu organizei o fluxo de trabalho para prever casos de diabetes com base nos dados disponíveis. A ideia foi deixar o processo mais limpo e fácil de acompanhar, sem misturar etapas de preparação, treino e avaliação de um jeito que pudesse causar vazamento de dados.

Eu concentrei as transformações dentro da validação cruzada para obter uma estimativa mais confiável do desempenho em dados novos. Assim, cada etapa acontece no momento correto e os resultados ficam mais consistentes.

## 1. Importações e configurações

Nesta primeira parte, eu reúno as bibliotecas usadas no notebook, junto com algumas configurações básicas do experimento. É aqui que também defino a chave de acesso, a URL do serviço e as colunas que realmente entram no modelo.

Eu deixei essas informações no começo para facilitar a organização e também para simplificar mudanças futuras. Se eu precisar ajustar alguma feature, trocar algum parâmetro ou reaproveitar o código em outro cenário, consigo localizar o que precisa ser alterado com mais facilidade.

In [ ]:
import os
import requests
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek

DEV_KEY = os.getenv("DEV_KEY", "SUA_CHAVE_AQUI")
URL = "https://aydanomachado.com/mlclass/01_Preprocessing.php"

FEATURE_COLS = ['Pregnancies', 'Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age']
VITALS_WITH_ZERO_AS_NAN = ['Glucose', 'BMI']
TARGET = 'Outcome'

print('Configurações carregadas.')
print(f'Features usadas: {FEATURE_COLS}')

Configurações carregadas.
Features usadas: ['Pregnancies', 'Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age']


## 2. Classes customizadas

Aqui eu criei uma transformação personalizada para ajustar o peso das variáveis de entrada. A proposta é simples: em vez de tratar todas as colunas exatamente da mesma forma, eu tento identificar quais delas trazem mais informação útil para a predição.

Para isso, eu combino duas ideias. A primeira é a mutual information, que me ajuda a medir o quanto cada feature se relaciona com o alvo. A segunda é a importância calculada por uma Random Forest, que também serve como uma forma de enxergar relevância entre as variáveis. Ao juntar esses dois sinais, eu consigo gerar pesos mais equilibrados e aplicar isso nos dados antes da etapa final de classificação.

Essa abordagem continua compatível com o pipeline e garante que a ponderação seja aprendida apenas a partir dos dados de treino em cada dobra da validação cruzada.

In [ ]:
class FeatureWeighter(BaseEstimator, TransformerMixin):
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.weights_ = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        mi = mutual_info_classif(X, y, random_state=self.random_state)
        rf = RandomForestClassifier(
            n_estimators=200,
            random_state=self.random_state,
            n_jobs=-1
        ).fit(X, y)

        weights = (mi + rf.feature_importances_) / 2.0
        weights = weights / (weights.mean() + 1e-9)
        self.weights_ = np.clip(weights, 0.2, None)
        return self

    def transform(self, X, y=None):
        X = np.asarray(X, dtype=float)
        return X * self.weights_

print('Classes customizadas prontas.')

Classes customizadas prontas.


## 3. Carga e preparação

Aqui eu leio os arquivos CSV e começo a preparação inicial dos dados. Primeiro, eu verifico rapidamente o tamanho de cada conjunto para conferir se os arquivos foram carregados corretamente e se o formato está dentro do esperado.

Depois disso, eu trato algumas colunas de forma especial. Em variáveis como Glucose e BMI, valores iguais a zero normalmente não fazem sentido como medida real, então eu os converto para ausentes. Isso evita que o modelo interprete esses zeros como valores válidos quando, na prática, eles representam informação faltante ou incorreta.

No fim dessa etapa, eu separo os dados de entrada e o alvo. Assim, o restante do notebook fica mais claro, porque cada parte passa a trabalhar diretamente com as variáveis que importam para o treino e para a previsão.

In [ ]:
print('Lendo os arquivos CSV...')
df_train = pd.read_csv('diabetes_dataset_raw.csv')
df_app = pd.read_csv('diabetes_app_raw.csv')

print(f'Treino: {df_train.shape[0]} linhas e {df_train.shape[1]} colunas')
print(f'Aplicação: {df_app.shape[0]} linhas e {df_app.shape[1]} colunas')

for col in VITALS_WITH_ZERO_AS_NAN:
    if col in df_train.columns:
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce').replace(0, np.nan)
    if col in df_app.columns:
        df_app[col] = pd.to_numeric(df_app[col], errors='coerce').replace(0, np.nan)

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET]
X_app = df_app[FEATURE_COLS]

print('Pré-processamento inicial concluído.')
print(f'Missing values no treino: {X_train.isna().sum().to_dict()}')

Lendo os arquivos CSV...
Treino: 572 linhas e 9 colunas
Aplicação: 196 linhas e 8 colunas
Pré-processamento inicial concluído.
Missing values no treino: {'Pregnancies': 0, 'Glucose': 5, 'BMI': 11, 'DiabetesPedigreeFunction': 0, 'Age': 0}


## 4. Pipeline e validação cruzada

Esta é a parte principal do processo, porque aqui eu reúno todas as transformações e o modelo final em uma única sequência. Isso garante que cada etapa seja aplicada na ordem correta e apenas com base nos dados de treino de cada dobra.

Primeiro entra o IterativeImputer, que estima os valores ausentes com base nas relações entre as variáveis. Em seguida, o RobustScaler e o QuantileTransformer ajudam a ajustar a escala e a distribuição dos dados, o que é especialmente importante para o KNN, já que ele depende diretamente de distâncias entre os pontos.

Depois disso, a classe FeatureWeighter ajusta a importância das colunas de acordo com os sinais aprendidos. Na sequência, o PCA reduz a dimensionalidade mantendo 90% da variância dos dados, o que ajuda a remover ruído e tornar o espaço mais compacto para o classificador.

Em seguida, o SMOTEENN trata o desbalanceamento das classes ao mesmo tempo em que realiza uma limpeza de exemplos mais ruidosos. Por fim, o KNN realiza a classificação com base na proximidade entre os exemplos.

Eu usei validação cruzada estratificada para avaliar o desempenho de forma mais consistente. Como ela preserva a proporção entre as classes em cada divisão, a estimativa tende a representar melhor o comportamento do modelo em dados novos. As métricas acompanhadas foram acurácia e recall, sendo o recall particularmente importante nesse contexto.

In [ ]:
from sklearn.preprocessing import RobustScaler
from imblearn.combine import SMOTEENN
from sklearn.decomposition import PCA

pipeline = Pipeline([
    ('imputer', IterativeImputer(random_state=42, max_iter=10)),
    ('scaler', RobustScaler()),
    ('qt', QuantileTransformer(n_quantiles=45, output_distribution='normal', random_state=42)),
    ('weighter', FeatureWeighter(random_state=42)),
    ('pca', PCA(n_components=0.90, random_state=42)),
    ('smote', SMOTEENN(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=3))
])

cv_strategy = RepeatedStratifiedKFold(n_splits=10, n_repeats=50, random_state=42)
print('Iniciando validação cruzada exaustiva (500 rodadas)...')

cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=['accuracy', 'recall'],
    n_jobs=-1
)

acc_mean = cv_results['test_accuracy'].mean() * 100
acc_std = cv_results['test_accuracy'].std() * 100
recall_mean = cv_results['test_recall'].mean() * 100
recall_std = cv_results['test_recall'].std() * 100

print('\n' + '='*35)
print('VALIDACÃO CRUZADA FINALIZADA')
print('='*35)
print(f'Acurácia Média: {acc_mean:.2f}% (+/- {acc_std:.2f}%)')
print(f'Recall Médio:   {recall_mean:.2f}% (+/- {recall_std:.2f}%)')
print('='*35)

Iniciando validação cruzada exaustiva (500 rodadas)...

VALIDACÃO CRUZADA FINALIZADA
Acurácia Média: 70.44% (+/- 6.07%)
Recall Médio:   80.76% (+/- 9.06%)


## 5. Treinamento final

Depois de avaliar o pipeline com a validação cruzada, eu treino o modelo final usando todo o conjunto de treino. Nesse ponto, o objetivo já não é mais medir desempenho, mas aproveitar a base completa para gerar previsões para os dados de aplicação.

Com o modelo ajustado, eu produzo as previsões para o conjunto final. Também mostro uma pequena amostra dessas previsões apenas para conferência, o que ajuda a verificar rapidamente se a saída está no formato esperado.

Por fim, eu organizo as predições em uma estrutura pronta para envio, junto com a chave de acesso exigida pela atividade. Assim, o resultado final já fica preparado para ser submetido sem precisar de ajustes extras.

In [ ]:
print('Treinando modelo final...')
pipeline.fit(X_train, y_train)

print('Gerando previsões...')
y_pred = pipeline.predict(X_app)

print(f'Total de previsões: {len(y_pred)}')
print(f'Primeiras previsões: {y_pred[:10]}')

dados_envio = {
    'dev_key': DEV_KEY,
    'predictions': pd.Series(y_pred).to_json(orient='values')
}

print('Estrutura pronta para envio.')

Treinando modelo final...
Gerando previsões...
Total de previsões: 196
Primeiras previsões: [0 1 1 0 1 1 1 1 1 0]
Estrutura pronta para envio.
